# core
> VPS provisioning: cloud-init generation, Hetzner hcloud CLI wrapper, SSH deployment helpers

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import getpass, json, os, shlex, subprocess, tempfile, time
from contextlib import contextmanager, suppress
from dockeasy import Cli, env_get, Compose, caddy_svc, cloudflared_svc, fasthtml_app
from fastcore.all import L, Path, run, listify, AttrDict, AttrDictDefault, delegates, modified_env
from fastcloudinit.core import cloud_init_base, cloud_init_config, user, runcmd, reboot
from hcloud import Client
from hcloud.images import Image
from hcloud.server_types import ServerType
from hcloud.locations import Location
from hcloud.ssh_keys import SSHKey

## Multipass for local testing
> Using `multipass` to test cloud-init and deployment locally before provisioning real VPSes. You'll need to install Multipass and have it in your PATH for this to work.

In [ ]:
#| export
class Multipass(Cli):
    'Wrap multipass CLI: manage local Ubuntu VMs for testing'
    def _run(self, cmd, *args): return run('multipass', cmd, *args)
    def launch(self,name,image='24.04',cpus=1,memory='1G',disk='10G',cloud_init:AttrDict=None, mounts=None) -> AttrDict:
        'Launch a VM. cloud_init: AttrDict(yaml, key) from multi_init() or vps_init(). Returns AttrDict(name, key).'
        args = ['multipass', 'launch', image, '-n', name, '-c', str(cpus), '-m', memory, '-d', disk]
        for hp, vp in (mounts or {}).items(): args += ['--mount', f'{hp}:{vp}']
        if cloud_init: args += ['--cloud-init', '-']
        subprocess.run(args, input=cloud_init.yaml if cloud_init else None, text=True, check=True)
        return AttrDict(name=name, key=cloud_init.key if cloud_init else None)

    def vms(self, running=False):
        'List VM names. running=True filters to Running state.'
        lst = L(json.loads(self('list', '--format', 'json')).get('list', []))
        if running: lst = lst.filter(lambda v: v.get('state') == 'Running')
        return list(lst.itemgot('name'))

    def ip(self, name) -> str:
        'Get IPv4 address of a VM.'
        return json.loads(self('info', name, '--format', 'json'))['info'][name]['ipv4'][0]

    def exec_(self, name, *cmd) -> str:
        'Run a command in a VM.'
        return self('exec', name, '--', *cmd)

    def rm(self, name, purge=True) -> None:
        'Delete a VM.'
        self('delete', '--purge' if purge else '', name)

mp = Multipass()

def deploy_mp(name, src, path='/srv/app', key=None, build=True, verbose=True) -> None:
    'Sync local directory into a Multipass VM and run docker compose up -d.'
    mp.exec_(name, 'sudo', 'install', '-d', '-o', 'deploy', '-g', 'deploy', path)
    deploy(mp.ip(name), src, path, key=key, name=name, build=build, verbose=verbose)

In [ ]:
Multipass().vms()

[]

## Cloud-init generation

`vps_init()` builds a cloud-init YAML for a fresh VPS via `fastcloudinit.cloud_init_config`: UFW hardening, user creation, SSH key setup, optional Docker. `multi_init()` is the local equivalent via `cloud_init_base` — same Docker setup, no UFW, no fail2ban.

In [ ]:
#| export
dock_cmd = ['curl -fsSL https://get.docker.com | sh', 'usermod -aG docker {username}','systemctl enable --now docker']
def keygen(path): return ['ssh-keygen', '-t', 'ed25519', '-f', str(path), '-N', '', '-q']
def _mk_kp(h,pub): return AttrDictDefault(pub_str=listify(pub)) if pub else gen_key(h)
def gen_key(slug, key_dir=None):
    'Generate ed25519 key pair at <key_dir>/<slug>. Overwrites existing. Returns AttrDict(key, pub, pub_str).'
    d = Path(key_dir or Path.home()/'.ssh')
    priv, pub = d/slug, d/f'{slug}.pub'
    subprocess.run(keygen(priv), input='y\n', text=True, capture_output=True, check=True)
    return AttrDict(key=priv, pub=pub, pub_str=[pub.read_text().strip()])

def vps_init(hostname, pub_keys=None, username='deploy', docker=True, pkgs=None, cmds=None, **kw):
    'Cloud-init for a fresh VPS. pub_keys=None → auto-generates ed25519 key pair. Returns AttrDict(yaml, key).'
    kp = _mk_kp(hostname, pub_keys)
    cmds = [c.format(username=username) for c in dock_cmd] if docker else [] + listify(cmds)
    pkgs = ['curl', 'fail2ban', 'unattended-upgrades'] + listify(pkgs)
    yaml = cloud_init_config(hostname=hostname,username=username,pub_keys=kp.pub_str,packages=pkgs,cmds=cmds,**kw)
    return AttrDict(yaml=yaml, key=kp.key)

In [ ]:
#| export
def multi_init(hostname, pub_keys=None, username='deploy', docker=True, pkgs=None, cmds=None):
    'Cloud-init for Multipass local VMs. pub_keys=None → auto-generates ed25519 key pair. Returns AttrDict(yaml, key).'
    kp = _mk_kp(hostname, pub_keys)
    cmds = [c.format(username=username) for c in dock_cmd] if docker else [] + listify(cmds)
    pkgs = ['curl'] + listify(pkgs)
    kw = {**runcmd(cmds), **reboot()} if cmds else {}
    yaml = cloud_init_base(hostname, packages=pkgs, users=[user(username, kp.pub_str)], **kw)
    return AttrDict(yaml=yaml, key=kp.key)

In [ ]:
# explicit pub_keys → no key generated, key=None
ci = multi_init('mylocal', 'ssh-rsa AAAA...')
assert '#cloud-config' in ci.yaml and 'get.docker.com' in ci.yaml and 'ufw' not in ci.yaml
assert 'power_state' in ci.yaml and ci.key is None
print(ci.yaml)

# auto-generate mode: key pair written to ~/.ssh/mylocal{,.pub}
ci2 = multi_init('mylocal', docker=False)
assert 'get.docker.com' not in ci2.yaml and 'power_state' not in ci2.yaml
assert ci2.key is not None and ci2.key.exists()
assert Path(str(ci2.key) + '.pub').exists()
ci2.key.unlink(); Path(str(ci2.key) + '.pub').unlink()
print('multi_init OK')

#cloud-config
hostname: mylocal
preserve_hostname: false
packages:
- curl
package_update: true
package_upgrade: true
disable_root: true
ssh_pwauth: false
users:
- name: deploy
  groups:
  - sudo
  shell: /bin/bash
  sudo:
  - ALL=(ALL) NOPASSWD:ALL
  ssh_authorized_keys:
  - ssh-rsa AAAA...
runcmd:
- curl -fsSL https://get.docker.com | sh
- usermod -aG docker deploy
- systemctl enable --now docker
power_state:
  mode: reboot
  message: Rebooting
  timeout: 1
  condition: true

multi_init OK


In [ ]:
ci = vps_init('myserver', 'ssh-rsa AAAA...', docker=True)
assert '#cloud-config' in ci.yaml and 'get.docker.com' in ci.yaml
assert 'fail2ban' in ci.yaml and 'unattended-upgrades' in ci.yaml
assert ci.key is None  # explicit pub_keys → no key generated
print(ci.yaml)
print('vps_init OK')

#cloud-config
hostname: myserver
preserve_hostname: false
packages:
- curl
- fail2ban
- unattended-upgrades
package_update: true
package_upgrade: true
disable_root: true
ssh_pwauth: false
users:
- name: deploy
  groups:
  - sudo
  shell: /bin/bash
  sudo:
  - ALL=(ALL) NOPASSWD:ALL
  ssh_authorized_keys:
  - ssh-rsa AAAA...
runcmd:
- curl -fsSL https://get.docker.com | sh
- usermod -aG docker deploy
- systemctl enable --now docker
- ufw default deny incoming
- ufw default allow outgoing
- ufw logging off
- ufw allow 22/tcp
- ufw --force enable
apt:
  conf: 'APT::Periodic::Update-Package-Lists "1";

    APT::Periodic::Download-Upgradeable-Packages "1";

    APT::Periodic::AutocleanInterval "7";

    APT::Periodic::Unattended-Upgrade "0";

    Unattended-Upgrade::Automatic-Reboot "false";

    '
write_files:
- path: /etc/logrotate.d/00-cloud-init-global
  owner: root:root
  permissions: '0644'
  content: "/var/log/*.log {\n    weekly\n    rotate 7\n    compress\n    su root adm\n    crea

## Hetzner (hcloud Python SDK)

`Hetzner` wraps the hcloud Python SDK — no CLI binary or config files required. Token read from `HCLOUD_TOKEN` by default.

In [ ]:
#| export
class Hetzner:
    'Hetzner Cloud VPS provider via hcloud Python SDK. Token read from HCLOUD_TOKEN by default.'
    def __init__(self, token=None):
        token = token or env_get('HCLOUD_TOKEN')
        if not token: raise ValueError('HCLOUD_TOKEN environment variable not set')
        self._c = Client(token=token)

    def servers(self) -> list:
        'List servers as [{name, ip, status}]'
        return L(self._c.servers.get_all()).map(lambda s: dict(name=s.name, ip=s.public_net.ipv4.ip, status=s.status))

    def server_ip(self, name) -> str:
        'Get public IPv4 of a server by name'
        s = self._c.servers.get_by_name(name)
        if not s: raise ValueError(f'Server {name!r} not found')
        return s.public_net.ipv4.ip

    def create(self, name, image='ubuntu-24.04', server_type='cx23', location=None, cloud_init=None, ssh_keys=None):
        '''Create a server. cloud_init: YAML string or AttrDict(yaml, key) from vps_init(). Returns AttrDict(ip, name, key, resp).
        NOTE: ssh_keys are Hetzner-registered keys injected into the root user only — they do NOT grant access to the
        deploy user created by cloud_init. Pass ssh_keys only when cloud_init is None (root-only access).'''
        if cloud_init is not None and ssh_keys:
            raise ValueError(
                'ssh_keys and cloud_init are conflicting strategies: ssh_keys grants root access only, '
                'while cloud_init creates a deploy user with its own key. '
                'Use cloud_init alone (key is in the returned AttrDict) or ssh_keys alone for root access.'
            )
        key = None
        if isinstance(cloud_init, AttrDict): key, cloud_init = cloud_init.key, cloud_init.yaml
        resp = self._c.servers.create(
            name=name,
            server_type=ServerType(name=server_type),
            image=Image(name=image),
            location=Location(name=location) if location else None,
            user_data=cloud_init,
            ssh_keys=[SSHKey(name=k) for k in (ssh_keys or [])],
        )
        return AttrDict(ip=resp.server.public_net.ipv4.ip, name=name, key=key, resp=resp)

    def delete(self, name) -> None:
        'Delete a server by name'
        s = self._c.servers.get_by_name(name)
        if s: s.delete()

    def keys(self) -> list:
        'List SSH keys as [{name, fingerprint}]'
        return L(self._c.ssh_keys.get_all()).map(lambda k: dict(name=k.name, fingerprint=k.fingerprint))

    def key_names(self) -> list:
        'Return SSH key name strings for use in create(ssh_keys=[...]) — root access only, not for deploy user'
        return [k['name'] for k in self.keys()]

### Integration tests

Requires `HCLOUD_TOKEN` in the environment. Creates a minimal `cx11` server, verifies the lifecycle, then deletes it.

In [ ]:
#| eval: False
hz = Hetzner()
L(hz.servers()).attrgot('name')

['vedicreader-cx32-hel']

In [ ]:
#| eval: False
sn = 'fastops-test'
hz.delete(sn)
svr=hz.create(sn, server_type='cx23', location='hel1')
assert svr.ip, 'create() should return an IP'
print(f'create OK: {svr.ip}')
o = lambda: L(hz.servers()).attrgot('name')
print(f'servers() OK: {o()}')
assert sn in o(), f'servers() should include {sn}'
assert hz.server_ip(sn) == svr.ip
hz.delete(sn)
assert sn not in o()
print(f'servers() after delete: {o()}')
print('delete() OK\nAll hcloud tests passed!')

create OK: 204.168.255.145
servers() OK: ['vedicreader-cx32-hel', 'lego', 'fastops-test']
servers() after delete: ['vedicreader-cx32-hel', 'lego']
delete() OK
All hcloud tests passed!


## SSH helpers

Pure subprocess-based SSH/rsync utilities — no paramiko dependency. `deploy()` syncs a Compose stack to a remote host and brings it up.

In [ ]:
#| export
def _ssh(host, user, key, port):
    a = ['ssh', '-o', 'StrictHostKeyChecking=accept-new']
    if key: a += ['-i', str(key)]
    if port != 22: a += ['-p', str(port)]
    return a + [f'{user}@{host}']

def _res_key(key=None, name=None):
    'Resolve SSH key: explicit path > name slug (~/.ssh/<name>) > None. Raise if file with slug missing.'
    if key: return str(key)
    if not name: return None
    p = Path.home()/'.ssh'/name
    if not p.exists(): raise FileNotFoundError(f'No SSH key at {p} — run gen_key({name!r}) first')
    print('Resolved SSH key from name slug:', p)
    return str(p)

def _resolve_pass(v, prompt):
    "Return v, or prompt via getpass if v is True."
    return getpass.getpass(prompt) if v is True else v

@contextmanager
def _askpass_env(pw):
    "Sets SSH_ASKPASS env for key passphrase or server password auth. No-op if pw is None."
    if not pw: yield; return
    fd, path = tempfile.mkstemp(prefix='.vpseasy_askpass_', suffix='.sh')
    try:
        os.write(fd, f'#!/bin/sh\necho {shlex.quote(pw)}\n'.encode())
        os.close(fd)
        os.chmod(path, 0o700)
        with modified_env('DISPLAY', SSH_ASKPASS=path, SSH_ASKPASS_REQUIRE='force'): yield
    finally:
        with suppress(FileNotFoundError): os.unlink(path)

def run_ssh(host, *cmds, user='deploy', key=None, name=None, port=22, key_pass=None, password=None, check=True, verbose=False, stdin_data=None):
    'Run commands on remote host via SSH. key_pass= for key passphrase, password= for server password auth. Pass True to either to prompt. stdin_data= bytes piped to remote stdin (e.g. for sudo -S).'
    key_pass = _resolve_pass(key_pass, 'SSH key passphrase: ')
    password = _resolve_pass(password, 'SSH password: ')
    ssh_cmd = _ssh(host, user, _res_key(key, name), port) + [' && '.join(cmds)]
    with _askpass_env(key_pass or password):
        res = subprocess.run(ssh_cmd, input=stdin_data, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    out = res.stdout.decode().strip()
    if res.stderr: out += ' ;; ' + res.stderr.decode().strip()
    if verbose: print(f'Ran SSH command on {host}: {" && ".join(cmds)} → {(res.returncode, out)}')
    if check and res.returncode: raise IOError(out)
    return (res.returncode, out) if not check else out


def sync(host, src='.', path='/srv/app', user='deploy', key=None, name=None, include=None, exclude=None,
         extra=None, key_pass=None, password=None, verbose=False):
    'Rsync local src to remote host:path. include= whitelist patterns, exclude= blacklist patterns. extra= extra rsync flags e.g. "--checksum" or ["--ignore-times","--partial"].'
    key_pass = _resolve_pass(key_pass, 'SSH key passphrase: ')
    password = _resolve_pass(password, 'SSH password: ')
    inner = f'mkdir -p {path} && chown {user}:{user} {path}'
    if password:
        a = f'[ -d {path} -a -w {path} ] || sudo -S sh -c {shlex.quote(inner)}'
        run_ssh(host, a, user=user, key=key, name=name, key_pass=key_pass, password=password, stdin_data=(password + chr(10)).encode())
    else:
        a = f'[ -d {path} -a -w {path} ] || sudo sh -c {shlex.quote(inner)}'
        run_ssh(host, a, user=user, key=key, name=name, key_pass=key_pass)
    if verbose: print(f'Ensured remote path {path} exists and is writable by {user}')
    ssh_e = ' '.join(_ssh(host, user, _res_key(key, name), 22)[:-1])
    inc, exc = listify(include), listify(exclude)
    cmd = ['rsync', '-az' + ('m' if inc else ''), '--delete', '-e', ssh_e, *listify(extra)]
    for p in exc: cmd += ['--exclude', p]
    if inc:
        for p in inc: cmd+=(['--include',p] if not p.endswith('/') else ['--include',p.rstrip('/'),'--include',p+'**'])
        cmd += ['--include', '*/', '--exclude', '*']
    cmd += [str(src).rstrip('/') + '/', f'{user}@{host}:{path}/']
    if verbose: print('Running rsync:', ' '.join(cmd))
    with _askpass_env(key_pass or password): subprocess.run(cmd)
    if verbose: print('Rsync completed successfully')

def chk_docker(host, u='deploy', k=None, name=None, key_pass=None, password=None, verbose=False) -> bool:
	'Verify docker daemon is running and user can run containers.'
	try:
		r = run_ssh(host, 'docker info', user=u, key=k, name=name, key_pass=key_pass, password=password)
		if verbose: print(f'Docker info: {r.strip()}')
		return True
	except Exception as e:
		if verbose: print(f'Docker check failed: {e}')
		return False

def _chk_compose(host, path, f='docker-compose.yml', u='deploy', k=None, name=None, key_pass=None, password=None, verbose=False) -> bool:
	'Check if Dockerfile exists in the remote path.'
	try:
		r = run_ssh(host, f'ls {path}/{f}', user=u, key=k, name=name, key_pass=key_pass, password=password)
		if verbose: print(f'docker-compose check output: {r.strip()}')
		return 'docker-compose' in r
	except Exception as e:
		if verbose: print(f'docker-compose check failed: {e}')
		return False

def deploy(host, src='.', path='/srv/app', user='deploy', build=True, key=None, name=None,
           include=None, exclude=None, extra=None, key_pass=None, password=None, verbose=False):
    'Sync src to host via rsync then docker compose up if docker is available. extra= extra rsync flags forwarded to sync().'
    key_pass = _resolve_pass(key_pass, 'SSH key passphrase: ')
    password = _resolve_pass(password, 'SSH password: ')
    sync(host, src, path, user, key=key, name=name, include=include, exclude=exclude, extra=extra, key_pass=key_pass, password=password, verbose=verbose)
    kw = dict(host=host, u=user, k=key, name=name, key_pass=key_pass, password=password, verbose=verbose)
    if not (chk_docker(**kw) and _chk_compose(path=path,**kw)): return
    a = f'cd {path} && docker compose up -d --remove-orphans' + (' --build' if build else '')
    r = run_ssh(host, a, user=user, key=key, name=name, key_pass=key_pass, password=password, verbose=verbose)
    if verbose: print('docker compose ran' + (' with build' if build else ''), '→', r.strip())

In [ ]:
# _ssh_base: pure flag construction
assert _ssh('1.2.3.4', 'deploy', None, 22) == ['ssh', '-o', 'StrictHostKeyChecking=accept-new', 'deploy@1.2.3.4']
assert _ssh('1.2.3.4', 'deploy', '/k', 2222)[-3:] == ['-p', '2222', 'deploy@1.2.3.4']
assert '-i' in _ssh('h', 'u', '/my/key', 22)
print('_ssh_base OK')
assert _res_key('/tmp/mykey') == '/tmp/mykey'
try: _res_key(name='__no_such_vpseasy_key__')
except FileNotFoundError as e: assert '__no_such_vpseasy_key__' in str(e)
print('_res_key OK')

_ssh_base OK
_res_key OK


In [ ]:
#| eval: False
r = hz._c.servers.get_by_name('vedicreader-cx32-hel')
run_ssh(r.public_net.ipv4.ip,user='vedicgit', password=True)

'Welcome to Ubuntu 24.04.2 LTS (GNU/Linux 6.8.0-60-generic x86_64)\n\n * Documentation:  https://help.ubuntu.com\n * Management:     https://landscape.canonical.com\n * Support:        https://ubuntu.com/pro\n\n System information as of Thu Jun  4 06:30:59 AM UTC 2026\n\n  System load:  0.23               Processes:             176\n  Usage of /:   34.2% of 74.79GB   Users logged in:       1\n  Memory usage: 62%                IPv4 address for eth0: 46.62.133.112\n  Swap usage:   0%                 IPv6 address for eth0: 2a01:4f9:c012:ef36::1\n\n * Strictly confined Kubernetes makes edge and IoT secure. Learn how MicroK8s\n   just raised the bar for easy, resilient and secure K8s cluster deployment.\n\n   https://ubuntu.com/engage/secure-kubernetes-at-the-edge\n\nExpanded Security Maintenance for Applications is not enabled.\n\n58 updates can be applied immediately.\nTo see these additional updates run: apt list --upgradable\n\nEnable ESM Apps to receive additional future security upda

## Verification helpers

Cloud-init runs asynchronously after `create()` returns. Use `wait_ssh()` to block until the server is reachable, then `check_cloud_init()` to confirm the bootstrap completed successfully before deploying.

In [ ]:
#| export
def wait_ssh(host, u='deploy', k=None, name=None, p=22, tout=300, interval=5, key_pass=None, password=None, verbose=True):
    'Poll SSH until connection succeeds or raises TimeoutError.'
    key_pass = _resolve_pass(key_pass, 'SSH key passphrase: ')
    password = _resolve_pass(password, 'SSH password: ')
    dl = time.time() + tout
    while time.time() < dl:
        try:
            run_ssh(host, 'true', user=u, key=k, name=name, port=p, key_pass=key_pass, password=password)
            if verbose: print(f'SSH to host {host} check succeeded')
            return True
        except Exception as e:
            if verbose: print(f'SSH to {host} not ready yet, retrying in {interval}s).Error:{e}')
            time.sleep(interval)
    raise TimeoutError(f'SSH to {host} not ready after {tout}s')

def chk_cloud_init(host, u='deploy', k=None, name=None, key_pass=None, password=None, verbose=True) -> str:
    'Return cloud-init status: done|running|error|unknown. check=False handles exit code 2 (done-with-warnings) on Ubuntu 24.04.'
    c = "test -f /var/lib/cloud/instance/boot-finished && echo 'status: done' || echo 'status: running'"
    o = run_ssh(host, c, user=u, key=k, name=name, key_pass=key_pass, password=password, check=False, verbose=verbose)
    o = o[1].strip()
    return o.split(': ', 1)[-1].strip() if ': ' in o else (o or 'unknown')


In [ ]:
# chk_docker: returns False when key lookup fails (FileNotFoundError caught internally — no network call)
assert chk_docker('localhost', name='__no_such_vpseasy_key__') is False
print('chk_docker: bad key → False OK')

# wait_ssh: raises TimeoutError immediately when tout=0
try: wait_ssh('1.2.3.4', tout=0); assert False
except TimeoutError as e: assert '1.2.3.4' in str(e)
print('wait_ssh: tout=0 → TimeoutError OK')

chk_docker: bad key → False OK
wait_ssh: tout=0 → TimeoutError OK


In [ ]:
#| export
def wait_ready(host,
               u='deploy',
               k=None,
               name=None,
               tout=300,
               interval=5,
               retries=2,
               key_pass=None,
               password=None,
               verbose=False
):
    'Wait for SSH then poll cloud-init until done. Retries cloud-init polling up to `retries` times before raising TimeoutError.'
    key_pass = _resolve_pass(key_pass, 'SSH key passphrase: ')
    password = _resolve_pass(password, 'SSH password: ')
    wait_ssh(host, u=u, k=k, name=name, tout=tout, interval=interval, key_pass=key_pass,
             password=password, verbose=verbose)
    for a in range(retries + 1):
        dl = time.time() + tout
        while time.time() < dl:
            status = chk_cloud_init(host, u=u, k=k, name=name, key_pass=key_pass, password=password)
            if verbose: print(f'cloud-init status: {status}')
            if status == 'done': return True
            if status == 'error': raise RuntimeError(f'cloud-init failed on {host}')
            time.sleep(interval)
        if a < retries: print(f'cloud-init on {host} still running after {tout}s — retrying ({a+1}/{retries}) ...')
    raise TimeoutError(f'cloud-init on {host} not done after {tout * (retries + 1)}s ({retries + 1} attempts)')

In [ ]:
#| export
def hetzner_deploy(name, # server name (also used for SSH key slug if key not given)
                   src,  # local path to sync and deploy
                   hz=None, # optional Hetzner() instance — creates one if not given
                   user='deploy', # remote username to deploy with (must match cloud-init user)
                   key=None, # optional SSH private key path or Path object (overrides name-based lookup of ~/.ssh/<name>)
                   image='ubuntu-24.04', # Hetzner image slug
                   server_type='cx23', # Hetzner server type slug
                   location=None, # Hetzner location slug e.g. 'hel1' (optional)
                   path='/srv/app', # remote path to sync to and deploy at
                   build=True, # whether to pass --build to docker compose up
                   include=None, # rsync include (whitelist) patterns
                   exclude=None, # rsync exclude (blacklist) patterns
                   extra=None, # extra rsync flags forwarded to sync() e.g. '--checksum' or ['--ignore-times','--partial']
                   key_pass=None, # passphrase for the SSH key, if it's encrypted
                   password=None, # server password for SSH auth (alternative to key-based auth)
                   tout=600, # seconds to wait for SSH and cloud-init each before retrying (default 10 minutes)
                   retries=2, # cloud-init retries before giving up and raising TimeoutError
                   verbose=True # whether to print verbose logs during wait and deploy steps
):
    'Full pipeline: provision Hetzner VPS (idempotent) → wait for cloud-init → deploy. Returns AttrDict(ip, name, key).'
    key_pass = _resolve_pass(key_pass, 'SSH key passphrase: ')
    password = _resolve_pass(password, 'SSH password: ')
    hz = hz or Hetzner()
    ex = hz._c.servers.get_by_name(name)
    if ex:
        ip, key = ex.public_net.ipv4.ip, _res_key(key=key, name=name)
        if verbose: print(f'Server {name} already exists at {ip}, checking cloud-init ...')
        wait_ready(ip, u=user, k=key, tout=tout, retries=retries, key_pass=key_pass, password=password, verbose=verbose)
    else:
        ci = vps_init(name)
        svr = hz.create(name, image=image, server_type=server_type, location=location, cloud_init=ci)
        ip, key = svr.ip, svr.key
        subprocess.run(['ssh-keygen', '-R', ip], capture_output=True)
        if verbose: print(f'Server {name} provisioning at {ip} ...')
        wait_ready(ip, u=user, k=key, tout=tout, retries=retries, key_pass=key_pass, password=password, verbose=verbose)
    deploy(ip, src, user=user, path=path, build=build, key=key, include=include, exclude=exclude,
           extra=extra, key_pass=key_pass, password=password, verbose=verbose)
    return AttrDict(ip=ip, name=name, key=key)

## SSH key helpers

`load_pub_keys()` resolves a list of paths (or auto-detects `~/.ssh/id_*.pub`) into a flat list of public key strings ready for `vps_init()` and `multi_init()`.

In [ ]:
#| export
def load_pub_keys(paths=None) -> list:
    'Load SSH public key strings. paths=None → auto-detect from ~/.ssh/id_*.pub'
    paths = paths or list(Path.home().glob('.ssh/id_*.pub'))
    return [Path(p).read_text().strip() for p in listify(paths) if Path(p).exists()]

In [ ]:
keys = load_pub_keys()
print(f'Found {len(keys)} local SSH key(s)')

_f = Path(tempfile.mktemp(suffix='.pub'))
_f.write_text('ssh-ed25519 TESTKEY comment')
assert load_pub_keys([str(_f)]) == ['ssh-ed25519 TESTKEY comment']
_f.unlink()
assert load_pub_keys(['/no_such_key.pub']) == []
print('load_pub_keys OK')

# gen_key: creates ed25519 pair, returns AttrDict, overwrites existing
_d = Path(tempfile.mkdtemp())
kp = gen_key('testkey', key_dir=_d)
assert kp.key.exists() and kp.pub.exists()
assert kp.pub_str[0].startswith('ssh-ed25519')
kp2 = gen_key('testkey', key_dir=_d)  # overwrite — should not raise
assert kp2.pub_str[0].startswith('ssh-ed25519')
import shutil; shutil.rmtree(_d)
print('gen_key OK')

Found 4 local SSH key(s)
load_pub_keys OK
gen_key OK


## Integration tests: SSH helpers and verification helpers

Requires Multipass. Launches a minimal Ubuntu VM with local SSH keys injected via `multi_init`, then exercises `wait_ssh`, `chk_cloud_init`, `chk_docker`, `run_ssh`, and `sync`.

In [ ]:
#| eval: False
# --- setup: auto-generate key pair, inject via multi_init, launch VM ---
_VM = 'testvm'
mp = Multipass()
try: mp.rm(_VM)
except: pass

ci = multi_init(_VM, docker=False) # generates ~/.ssh/fastops-vpstest{,.pub}
vm = mp.launch(_VM, image='24.04', cpus=1, memory='512M', disk='5G', cloud_init=ci)
ip = mp.ip(vm.name)
_key = vm.key
print(f'VM at {ip}, key: {_key}')

Creating testvm  Configuring testvm  Starting testvm  Waiting for initialization to complete  Launched: testvm
VM at 192.168.2.64, key: /Users/71293/.ssh/testvm


In [ ]:
#| eval: False
# wait_ssh / chk_cloud_init / run_ssh
assert wait_ssh(ip, k=_key, tout=30) is True; print('wait_ssh OK')
status = chk_cloud_init(ip, name=_VM, verbose=True)
assert status in ('done', 'running'), f'unexpected: {status!r}'; print(f'chk_cloud_init: {status}')
assert run_ssh(ip, 'echo hi', key=_key).strip() == 'hi'; print('run_ssh OK')

SSH to host 192.168.2.64 check succeeded
wait_ssh OK
Resolved SSH key from name slug: /Users/71293/.ssh/testvm
Ran SSH command on 192.168.2.64: test -f /var/lib/cloud/instance/boot-finished && echo 'status: done' || echo 'status: running' → (0, 'status: done')
chk_cloud_init: done
run_ssh OK


In [ ]:
#| eval: False
# sync: full dir / exclude / include (whitelist)
_src = Path(tempfile.mkdtemp())
(_src/'a.txt').write_text('hello')
(_src/'b.log').write_text('log')
(_src/'sub').mkdir(); (_src/'sub'/'c.txt').write_text('sub')

deploy_mp(_VM, _src, path='/tmp/app')  # sanity check deploy_mp with build=False
assert run_ssh(ip, 'ls /tmp/app', key=_key).split() == ['a.txt', 'b.log', 'sub']

sync(ip, _src, '/tmp/t1', key=_key)
assert run_ssh(ip, 'cat /tmp/t1/a.txt', key=_key).strip() == 'hello'; print('sync full OK')

sync(ip, _src, '/tmp/t2', key=_key, exclude=['*.log'])
assert run_ssh(ip, 'ls /tmp/t2', key=_key).split() == ['a.txt', 'sub']; print('sync exclude OK')

sync(ip, _src, '/tmp/t3', key=_key, include=['a.txt'])
assert run_ssh(ip, 'ls /tmp/t3', key=_key).strip() == 'a.txt'; print('sync include OK')

sync(ip, _src, '/tmp/t4', key=_key, include=['sub/'])
assert run_ssh(ip, 'cat /tmp/t4/sub/c.txt', key=_key).strip() == 'sub'; print('sync include-dir OK')

Resolved SSH key from name slug: /Users/71293/.ssh/testvm
Ensured remote path /tmp/app exists and is writable by deploy
Resolved SSH key from name slug: /Users/71293/.ssh/testvm
Running rsync: rsync -az --delete -e ssh -o StrictHostKeyChecking=accept-new -i /Users/71293/.ssh/testvm /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp25j4jzmr/ deploy@192.168.2.64:/tmp/app/
Rsync completed successfully
Resolved SSH key from name slug: /Users/71293/.ssh/testvm
Docker check failed:  ;; bash: line 1: docker: command not found
sync full OK
sync exclude OK
sync include OK
sync include-dir OK


In [ ]:
#| eval: False
# extra= forwards arbitrary rsync flags. str and list both accepted via listify().
sync(ip, _src, '/tmp/t5', key=_key, extra='--checksum')
assert run_ssh(ip, 'cat /tmp/t5/a.txt', key=_key).strip() == 'hello'; print('sync extra=str (--checksum, hash-match) OK')

sync(ip, _src, '/tmp/t6', key=_key, extra=['--ignore-times', '--partial'])
assert run_ssh(ip, 'cat /tmp/t6/a.txt', key=_key).strip() == 'hello'; print('sync extra=list (--ignore-times re-copies all) OK')

# extra= composes with include/exclude
sync(ip, _src, '/tmp/t7', key=_key, extra='--checksum', exclude=['*.log'])
assert run_ssh(ip, 'ls /tmp/t7', key=_key).split() == ['a.txt', 'sub']; print('sync extra + exclude OK')

sync extra=str (--checksum, hash-match) OK
sync extra=list (--ignore-times re-copies all) OK
sync extra + exclude OK


In [ ]:
#| eval: False
# --- teardown ---
mp.rm(_VM); print('VM removed')

VM removed


## Docker Compose helpers

`vols_to_binds()` converts absolute container paths to local bind mounts. `caddy_stack()` generates a full production Compose file: `app` + `caddy` + optional `cloudflared`, `web` network, and caddy volumes — one call replaces the boilerplate in every FastHTML deploy.

In [ ]:
#| export
def vols_to_binds(vols):
    'Convert ["/app/data"] → ["./data:/app/data"] for docker compose bind mounts'
    return [f'./{v.split("/")[-1]}:{v}' for v in listify(vols)]

def caddy_stack(domain, df, vols=None, env_file='.env', cloudflared=True, root=None, conf=None, **kw):
    '''Compose with app + caddy + optional cloudflared, web network, caddy volumes.
    df: Dockerfile instance. root: if given, saves Dockerfile + docker-compose.yml there. **kw passed to caddy_svc.'''
    root = Path(root if root else '.')
    df.save(Path(root)/'Dockerfile')
    v,env = vols_to_binds(vols) if vols else None, listify(env_file) if env_file else None
    c = (Compose()
         .svc('app', build=df, volumes=v, env_file=env, restart='unless-stopped', networks=['web'])
         .svc('caddy', **caddy_svc(domain, cloudflared=cloudflared, conf=conf or root/'Caddyfile', **kw)))
    if cloudflared: c = c.svc('cloudflared', **cloudflared_svc(url='http://caddy'))
    c = c.network('web').volume('caddy_data').volume('caddy_config')
    if root: c.save(root/'docker-compose.yml')
    return c

In [ ]:
assert vols_to_binds(['/app/data', '/app/backups']) == ['./data:/app/data', './backups:/app/backups']
assert vols_to_binds('/app/data') == ['./data:/app/data']
print('vols_to_binds OK')

_d = Path(tempfile.mkdtemp())
c = caddy_stack('myapp.example.com', fasthtml_app(), vols=['/app/data'], root=_d, conf=str(_d/'Caddyfile'))
d = c.to_dict()
assert set(d['services']) == {'app', 'caddy', 'cloudflared'}
assert 'web' in d['networks']
assert 'caddy_data' in d['volumes'] and 'caddy_config' in d['volumes']
assert d['services']['app']['volumes'] == ['./data:/app/data']
assert (_d/'Dockerfile').exists() and (_d/'docker-compose.yml').exists() and (_d/'Caddyfile').exists()
print('caddy_stack OK')

vols_to_binds OK
caddy_stack OK


In [ ]:
caddy_stack('myapp.example.com', fasthtml_app(), vols=['/app/data'])

services:
  app:
    build: .
    volumes:
    - ./data:/app/data
    env_file:
    - .env
    restart: unless-stopped
    networks:
    - web
  caddy:
    image: caddy:2
    depends_on:
    - app
    volumes:
    - ./Caddyfile:/etc/caddy/Caddyfile
    - caddy_data:/data
    - caddy_config:/config
    networks:
    - web
    restart: unless-stopped
  cloudflared:
    image: cloudflare/cloudflared:latest
    command: tunnel --no-autoupdate run --url http://caddy
    environment:
    - TUNNEL_TOKEN=${CF_TUNNEL_TOKEN}
    networks:
    - web
    restart: unless-stopped
networks:
  web: null
volumes:
  caddy_data: null
  caddy_config: null

In [ ]:
#| export
def mv_skill_md(dry_run=True, dir=None) -> None:
    'Copy bundled SKILL.md to .agents/skills/vpseasy/ and ~/.claude/skills/vpseasy/'
    base = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    src = base/'SKILL.md'
    if not src.exists(): return
    root = Path(dir or '.')
    ts = [root/'.agents/skills/vpseasy/SKILL.md',
          Path.home()/'.claude/skills/vpseasy/SKILL.md']
    if dry_run: print(f'Would copy to: {[str(p) for p in ts]}')
    else: [p.mk_write(src.read_text(encoding='utf-8')) for p in ts]
    if not dry_run: print(f'Installed → {[str(p) for p in ts]}')

In [ ]:
import io, sys, tempfile, os
_d = Path(tempfile.mkdtemp())
(_d/'SKILL.md').write_text('test-skill')   # provide a src so function doesn't bail early
_saved_cwd = os.getcwd(); os.chdir(_d)
_buf = io.StringIO(); sys.stdout, _saved = _buf, sys.stdout
mv_skill_md(dry_run=True, dir=str(_d))
sys.stdout = _saved; os.chdir(_saved_cwd)
_out = _buf.getvalue()
assert '.agents/skills/vpseasy/SKILL.md' in _out, _out
assert '.claude/skills/vpseasy/SKILL.md' in _out, _out
print('mv_skill_md dry_run OK')

mv_skill_md dry_run OK


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()